###Step 0: Configurações iniciais

Importação de bibliotecas, funções, etc. Definições de escopo, nomes de tabelas, path's, etc.

In [0]:
import requests
import json
import os
import re
from pyspark.sql import functions as F, Window as W
from pyspark.sql.functions import from_json, col
from pyspark.sql.functions import udf, col
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from functools import reduce
from pyspark.sql import DataFrame, Row
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta

from pyspark.sql.functions import date_sub, current_date, date_format
from delta.tables import DeltaTable
from pyspark.sql.types import *

In [0]:
# #Uso para desenvolvimento e testes

scope_api="production"


table_name="SD3"
table = {
    "catalog": "ihara_datalake_incremental",
    "schema": "raw",
    "table_name": f"{table_name}",
    "file_path": f"schema_table/totvs/{table_name}.json"
  }

sink = f"{table['catalog']}.{table['schema']}.{table['table_name']}"
df = spark.read.json(f"/Volumes/{table['catalog']}/raw/{table['file_path']}")

# Coleta a primeira linha do DataFrame e a converte para um dicionário
table = df.collect()[0].asDict()
table


In [0]:

scope_api="production"

# Obtém o valor do widget "catalog" e armazena na variável catalog
catalog = dbutils.widgets.get("catalog")

# Obtém o valor do widget "path_files" e armazena na variável path_files
path_files = dbutils.widgets.get("path_files")

# Obtém o valor do widget "table" e armazena na variável table_raw
table_raw = dbutils.widgets.get("table")

# Lê o arquivo JSON localizado no caminho especificado por catalog e path_files e armazena em um DataFrame
df = spark.read.json(f"/Volumes/{catalog}/{path_files}")

# Coleta a primeira linha do DataFrame e a converte para um dicionário
table = df.collect()[0].asDict()

# Cria a string de destino no formato "catalog.raw.table_raw"
sink = f"{catalog}.raw.{table_raw}"



###Step 1: Obtenção das secrets
Obtenção das secrets de acesso a base de dados Oracle

In [0]:

oracle_port = "1521"
oracle_host = dbutils.secrets.get(scope_api, 'totvs_oracle_host')
oracle_service_name = dbutils.secrets.get(scope_api, 'totvs_oracle_service_name')
oracle_user = dbutils.secrets.get(scope_api, 'totvs_oracle_user')
oracle_password = dbutils.secrets.get(scope_api, 'totvs_oracle_password')



###Step 2: Extração dos dados Totvs
Obter a extração das tabelas Totvs através da base de dados Oracle, e porteriormente salvando em um Dataframe

In [0]:
def read_from_oracle(oracle_host: str, oracle_port: str, oracle_service_name: str, oracle_user: str, oracle_password: str, query: str) -> DataFrame:
    jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service_name}"
    connection_properties = {
        "user": oracle_user,
        "password": oracle_password,
        "driver": "oracle.jdbc.OracleDriver",
        "fetchsize": "1000"
    }
    query = "(" + query + ")"
    df = spark.read.jdbc(
        url=jdbc_url,
        table=query,
        properties=connection_properties
    )
    return df


###Step 2: Processamento de dados
Este bloco de código contém a logica de carga tanto incremental coma a full.Utilizando duas funções: load_incremental e load_full

In [0]:

def load_incremental(tabela, field_date, period_date):

  
    # Load Delta table
    delta_table = DeltaTable.forName(spark, sink)

    # Define condition for deletion
    condition = f"{field_date} >= date_format(date_sub(current_date(), {period_date}), 'yyyyMMdd')"

    # Delete rows based on condition
    delta_table.delete(condition)

    
    #Query dinâmica para fazer leitura dos dados no Oracle baseado nos parâmetros de entrada
    query = f"""(SELECT * FROM {tabela}010 WHERE D_E_L_E_T_ = ' '
    AND TO_DATE({field_date}, 'YYYYMMDD') >= TRUNC(SYSDATE - {period_date}))"""

    #Load dados da base Orcle Totvs
    df = read_from_oracle(oracle_host, oracle_port, oracle_service_name, oracle_user, oracle_password, query)
    

    df = df.withColumn("dh_insercao_raw", F.from_utc_timestamp(F.current_timestamp(), "Brazil/East").cast("timestamp"))

    # Escrever dados em Delta Lake
    df.write \
      .format("delta") \
      .mode("append") \
      .option("mergeSchema", "true") \
      .option('overwriteSchema', 'true') \
      .saveAsTable(sink)

    # Log
    print(f"Tabela {tabela} processada com sucesso!")
  





In [0]:
def load_full(tabela):
  
    print(f"Processando tabela: {tabela}")

  
    # Construir query de leitura
    query = f"SELECT * FROM {tabela}010 WHERE D_E_L_E_T_ =' '"

    if (tabela=='SYS_COMPANY'):
        query = f"SELECT * FROM {tabela} WHERE D_E_L_E_T_ =' '"
    
    df = read_from_oracle(oracle_host, oracle_port, oracle_service_name, oracle_user, oracle_password, query)

    # Adicionar colunas de controle
    df = df.withColumn("dh_insercao_raw", F.from_utc_timestamp(F.current_timestamp(), "Brazil/East").cast("timestamp"))

    # Escrever dados em Delta Lake
    (df.write \
   .format('delta') \
   .option('mergeSchema', 'true') \
   .option('overwriteSchema', 'true') \
   .saveAsTable(sink, mode='overwrite'))

    
    print(f"Tabela {tabela} processada com sucesso!")
    print(sink)
    print(query)


In [0]:
def __is_first_load(sink):
  try:
   spark.table(sink)

  except:
   return True

###Step 3: Gravar os dados no catalog (RAW)

Gravar os dados baseado no tipo de carga(full/incremental) que foi definido no parâmetro

In [0]:
if __is_first_load(sink):
  
  load_full(table["table_name"])
  dbutils.notebook.exit('First load')


In [0]:
# load_full(table["table_name"])


In [0]:

if (table["load_type"]=="full"):
     
     load_full(table["table_name"])
else:
     load_incremental(table["table_name"],table["field_date"],table["period_date"])